# TransReID Semantic Demo

Pipeline: setup -> sanity check -> optional semantic-mask prep -> merged config -> smoke forward -> train/test commands -> retrieval visualization.

In [ ]:
import os, sys, math, shlex, subprocess, re
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

def find_root(start=None):
    p = Path.cwd().resolve() if start is None else Path(start).resolve()
    for c in [p] + list(p.parents):
        if (c/'train.py').exists() and (c/'config'/'defaults.py').exists():
            return c
        if (c/'semantic'/'train.py').exists() and (c/'semantic'/'config'/'defaults.py').exists():
            return c/'semantic'
    raise FileNotFoundError('semantic repo root not found')

ROOT = find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('ROOT =', ROOT)
print('Python =', sys.version.split()[0])
print('Torch =', torch.__version__)
print('CUDA =', torch.cuda.is_available())

DATA_ROOT = Path('/content/data')
DATASET = 'market1501'
DATASET_ROOT = DATA_ROOT / DATASET
PRETRAIN = Path('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/checkpoint/jx_vit_base_p16_224-80ecf9dd.pth')
OUT = Path('/content/drive/MyDrive/TGMT/PROJECT/Colab/semantic_demo_logs/market_semantic_demo')
CFG_MAP = {
    'baseline': 'configs/Market/vit_transreid_stride.yml',
    'reliability': 'configs/Market/vit_transreid_stride_local_reliability.yml',
    'semantic_reliability': 'configs/Market/vit_transreid_stride_sem_decoder_reliability.yml',
}
CFG_NAME = 'semantic_reliability'
CFG_FILE = Path(CFG_MAP[CFG_NAME])
TEST_WEIGHT = None
RUN_PREPARE = False
RUN_TRAIN = False
RUN_TEST = False
RUN_SMOKE = True
DEVICE_ID = '0'
IMS = 16
WORKERS = 2
EPOCHS = 5
EVAL_PERIOD = 1
CKPT_PERIOD = 1
QUERY_INDEX = 0
TOPK = 5
MASK_DIR = DATASET_ROOT / 'semantic_groups'
RAW_MASK_DIR = DATASET_ROOT / 'semantic_raw'
EXTRA_OPTS = [
    'MODEL.DEVICE_ID', DEVICE_ID,
    'MODEL.PRETRAIN_CHOICE', 'imagenet',
    'MODEL.PRETRAIN_PATH', str(PRETRAIN),
    'DATASETS.ROOT_DIR', str(DATA_ROOT),
    'SOLVER.IMS_PER_BATCH', str(IMS),
    'DATALOADER.NUM_WORKERS', str(WORKERS),
    'SOLVER.MAX_EPOCHS', str(EPOCHS),
    'SOLVER.EVAL_PERIOD', str(EVAL_PERIOD),
    'SOLVER.CHECKPOINT_PERIOD', str(CKPT_PERIOD),
    'OUTPUT_DIR', str(OUT),
]
print('CFG_FILE =', CFG_FILE)
print('DATASET_ROOT =', DATASET_ROOT)
print('OUT =', OUT)

In [ ]:
# Optional installs:
# %pip install -r requirements.txt
# %pip install transformers accelerate tqdm scikit-learn

def list_imgs(folder, n=6):
    folder = Path(folder)
    exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    return [p for p in sorted(folder.iterdir()) if p.suffix.lower() in exts][:n] if folder.exists() else []

def show_imgs(paths, title, cols=3):
    if not paths:
        print(title, ': empty')
        return
    rows = math.ceil(len(paths)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axes = np.array(axes).reshape(rows, cols)
    for ax in axes.flat: ax.axis('off')
    for ax, p in zip(axes.flat, paths):
        ax.imshow(Image.open(p).convert('RGB')); ax.set_title(p.name, fontsize=8); ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

PAL = np.array([[0,0,0],[239,71,111],[255,209,102],[6,214,160],[17,138,178],[7,59,76],[131,56,236]], dtype=np.uint8)
def color_mask(m): return PAL[np.clip(np.asarray(m, dtype=np.int64), 0, len(PAL)-1)]
def overlay(img, mask, a=0.45):
    img = np.asarray(img).astype(np.float32)
    m = np.asarray(mask)
    mrgb = color_mask(m).astype(np.float32)
    valid = (m > 0).astype(np.float32)[..., None]
    return (img * (1 - a*valid) + mrgb * (a*valid)).astype(np.uint8)

show_imgs(list_imgs(DATASET_ROOT/'bounding_box_train'), 'Train samples')
show_imgs(list_imgs(DATASET_ROOT/'query'), 'Query samples')
show_imgs(list_imgs(DATASET_ROOT/'bounding_box_test'), 'Gallery samples')

qs = list_imgs(DATASET_ROOT/'query', 3)
if qs:
    fig, axes = plt.subplots(len(qs), 3, figsize=(12, 4*len(qs)))
    axes = np.atleast_2d(axes)
    for r, p in enumerate(qs):
        img = Image.open(p).convert('RGB')
        mp = (MASK_DIR / p.relative_to(DATASET_ROOT)).with_suffix('.png')
        axes[r,0].imshow(img); axes[r,0].set_title(p.name); axes[r,0].axis('off')
        if mp.exists():
            mask = np.array(Image.open(mp), dtype=np.uint8)
            axes[r,1].imshow(color_mask(mask)); axes[r,2].imshow(overlay(img, mask))
        else:
            axes[r,1].text(0.5,0.5,'mask missing',ha='center',va='center'); axes[r,2].text(0.5,0.5,'mask missing',ha='center',va='center')
        axes[r,1].set_title('mask'); axes[r,2].set_title('overlay'); axes[r,1].axis('off'); axes[r,2].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
prep = [sys.executable, 'tools/prepare_semantic_maps.py', '--dataset-root', str(DATASET_ROOT), '--output-root', str(MASK_DIR), '--raw-output-root', str(RAW_MASK_DIR), '--model-id', 'fashn-ai/fashn-human-parser', '--preset', 'fashn6', '--batch-size', '8', '--device', 'cuda' if torch.cuda.is_available() else 'cpu']
print('Prepare cmd:')
print(' '.join(shlex.quote(x) for x in prep))
if RUN_PREPARE:
    print('Return code =', subprocess.run(prep, cwd=ROOT, text=True).returncode)
else:
    print('RUN_PREPARE = False')

from config import cfg as base_cfg
def build_cfg(cfg_file, opts):
    c = base_cfg.clone(); c.merge_from_file(str(cfg_file)); c.merge_from_list(list(opts)); c.freeze(); return c
cfg = build_cfg(CFG_FILE, EXTRA_OPTS)
print(cfg)
print('Flags:', {'JPM': cfg.MODEL.JPM, 'REL': cfg.MODEL.RELIABILITY_PIPELINE, 'SEM': cfg.MODEL.SEM_ALIGN.ENABLED, 'PIXEL_DEC': cfg.MODEL.SEM_ALIGN.PIXEL_DECODER_ENABLED, 'TOKEN_ENRICH': cfg.MODEL.TOKEN_ENRICH.ENABLED, 'VIS_WEIGHT': cfg.MODEL.VIS_WEIGHT.ENABLED})

In [ ]:
from datasets import make_dataloader
from datasets.make_dataloader import ReIDPairTransform
from model import make_model
from utils.metrics import euclidean_distance, eval_func

train_loader, train_loader_normal, val_loader, num_query, num_classes, camera_num, view_num = make_dataloader(cfg)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = make_model(cfg, num_class=num_classes, camera_num=camera_num, view_num=view_num).to(device).eval()
if TEST_WEIGHT:
    st = torch.load(TEST_WEIGHT, map_location='cpu'); st = st['model'] if isinstance(st, dict) and 'model' in st else st; model.load_state_dict(st, strict=False)

def denorm(x):
    m = torch.tensor(cfg.INPUT.PIXEL_MEAN, dtype=x.dtype).view(-1,1,1)
    s = torch.tensor(cfg.INPUT.PIXEL_STD, dtype=x.dtype).view(-1,1,1)
    return (x.cpu()*s + m).clamp(0,1).permute(1,2,0).numpy()

def up_heat(v, grid, size):
    t = torch.as_tensor(v).float().reshape(1,1,grid[0],grid[1])
    return F.interpolate(t, size=size, mode='bilinear', align_corners=False)[0,0].cpu().numpy()

if RUN_SMOKE:
    batch = next(iter(train_loader))
    if len(batch) == 5: imgs, pids, camids, viewids, sem = batch
    else: imgs, pids, camids, viewids = batch; sem = None
    with torch.no_grad():
        fd = model(imgs.to(device), cam_label=camids.to(device), view_label=viewids.to(device), return_feature_dict=True, semantic_masks=sem.to(device) if sem is not None else None)
    print({k: tuple(v.shape) if torch.is_tensor(v) else type(v).__name__ for k, v in fd.items() if k in ['retrieval_feat','patch_tokens','patch_weights','patch_reliability','semantic_pixel_logits','local_weights']})
    grid = fd.get('patch_grid', (21, 10))
    vis = [('image', denorm(imgs[0]))]
    if sem is not None: vis.append(('gt_mask', color_mask(sem[0].numpy().astype(np.uint8))))
    if fd.get('semantic_pixel_logits') is not None: vis.append(('pred_mask', color_mask(fd['semantic_pixel_logits'][0].argmax(0).cpu().numpy().astype(np.uint8))))
    if fd.get('patch_weights') is not None: vis.append(('patch_weights', up_heat(fd['patch_weights'][0], grid, cfg.INPUT.SIZE_TRAIN)))
    if fd.get('patch_reliability') is not None: vis.append(('patch_reliability', up_heat(fd['patch_reliability'][0], grid, cfg.INPUT.SIZE_TRAIN)))
    if fd.get('selection_mask') is not None: vis.append(('selection_mask', up_heat(fd['selection_mask'][0], grid, cfg.INPUT.SIZE_TRAIN)))
    fig, axes = plt.subplots(1, len(vis), figsize=(4*len(vis), 4))
    axes = np.atleast_1d(axes)
    for ax, (name, arr) in zip(axes, vis):
        ax.imshow(arr if arr.ndim == 3 else arr, cmap=None if arr.ndim == 3 else 'magma'); ax.set_title(name); ax.axis('off')
    plt.tight_layout(); plt.show()
    if fd.get('local_weights') is not None:
        lw = fd['local_weights'][0].cpu().numpy(); plt.figure(figsize=(6,4)); plt.bar(np.arange(len(lw)), lw); plt.xticks(np.arange(len(lw)), [f'JPM-{i+1}' for i in range(len(lw))]); plt.title('local weights'); plt.show()
else:
    print('RUN_SMOKE = False')

In [ ]:
def run_cmd(cmd):
    print(' '.join(shlex.quote(str(x)) for x in cmd))
    p = subprocess.Popen([str(x) for x in cmd], cwd=str(ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end=''); lines.append(line)
    rc = p.wait(); print('\nReturn code =', rc); return rc, ''.join(lines)

OUT.mkdir(parents=True, exist_ok=True)
train_cmd = [sys.executable, 'train.py', '--config_file', str(CFG_FILE)] + EXTRA_OPTS
test_path = TEST_WEIGHT
if test_path is None:
    for cand in [OUT/'transformer_best.pth', OUT/'latest_model.pth', OUT/'best_model.pth']:
        if cand.exists(): test_path = cand; break
test_cmd = [sys.executable, 'test.py', '--config_file', str(CFG_FILE)] + EXTRA_OPTS + (['TEST.WEIGHT', str(test_path)] if test_path else [])
print('Train cmd:'); print(' '.join(shlex.quote(str(x)) for x in train_cmd))
print('Test cmd:'); print(' '.join(shlex.quote(str(x)) for x in test_cmd))
if RUN_TRAIN: run_cmd(train_cmd)
if RUN_TEST and test_path: run_cmd(test_cmd)

logp = OUT/'train_log.txt'
if logp.exists():
    maps = []; cur = None
    for line in logp.read_text(encoding='utf-8', errors='ignore').splitlines():
        m = re.search(r'Validation Results - Epoch:\s*(\d+)', line)
        if m: cur = int(m.group(1)); continue
        m = re.search(r'mAP:\s*([0-9.]+)%', line)
        if m and cur is not None: maps.append((cur, float(m.group(1))))
    if maps:
        x, y = zip(*maps); plt.figure(figsize=(6,4)); plt.plot(x, y, marker='o'); plt.title('validation mAP'); plt.xlabel('epoch'); plt.ylabel('mAP %'); plt.grid(alpha=0.3); plt.show()

In [ ]:
if test_path and Path(test_path).exists():
    eval_model = make_model(cfg, num_class=num_classes, camera_num=camera_num, view_num=view_num).to(device).eval()
    st = torch.load(test_path, map_location='cpu'); st = st['model'] if isinstance(st, dict) and 'model' in st else st; eval_model.load_state_dict(st, strict=False)
    feats = []; pids = []; cams = []; paths = []
    for imgs, pid_batch, camid_list, camids_batch, viewids_batch, path_batch in val_loader:
        with torch.no_grad():
            feat = eval_model(imgs.to(device), cam_label=camids_batch.to(device), view_label=viewids_batch.to(device))
        feats.append(feat.cpu()); pids.extend(list(pid_batch)); cams.extend(list(camid_list)); paths.extend(list(path_batch))
    feats = torch.cat(feats, 0)
    if cfg.TEST.FEAT_NORM == 'yes': feats = F.normalize(feats, dim=1)
    qf, gf = feats[:num_query], feats[num_query:]
    qp, gp = np.asarray(pids[:num_query]), np.asarray(pids[num_query:])
    qc, gc = np.asarray(cams[:num_query]), np.asarray(cams[num_query:])
    qpaths, gpaths = paths[:num_query], paths[num_query:]
    dist = euclidean_distance(qf, gf)
    cmc, mAP = eval_func(dist, qp, gp, qc, gc)
    print('Notebook eval: mAP = %.4f Rank-1 = %.4f' % (mAP, cmc[0]))
    order = np.argsort(dist[QUERY_INDEX])
    keep = []
    for gi in order:
        if gp[gi] == qp[QUERY_INDEX] and gc[gi] == qc[QUERY_INDEX]:
            continue
        keep.append(gi)
        if len(keep) >= TOPK: break
    fig, axes = plt.subplots(1, TOPK + 1, figsize=(3*(TOPK+1), 4))
    axes[0].imshow(Image.open(qpaths[QUERY_INDEX]).convert('RGB')); axes[0].set_title(f'Query\nPID={qp[QUERY_INDEX]}'); axes[0].axis('off')
    for ax, gi, rank in zip(axes[1:], keep, range(1, len(keep)+1)):
        ok = gp[gi] == qp[QUERY_INDEX]
        ax.imshow(Image.open(gpaths[gi]).convert('RGB')); ax.set_title(f'R{rank}\nPID={gp[gi]}\n{"MATCH" if ok else "WRONG"}'); ax.axis('off')
        for sp in ax.spines.values(): sp.set_visible(True); sp.set_linewidth(4); sp.set_edgecolor('limegreen' if ok else 'crimson')
    plt.tight_layout(); plt.show()
    qimg = Path(qpaths[QUERY_INDEX]); qmask = (MASK_DIR / qimg.relative_to(DATASET_ROOT)).with_suffix('.png')
    tfm = ReIDPairTransform(cfg, is_train=False)
    img = Image.open(qimg).convert('RGB'); mask = Image.open(qmask).convert('L') if qmask.exists() else None
    img_t, mask_t = tfm(img, mask)
    with torch.no_grad():
        qfd = eval_model(img_t.unsqueeze(0).to(device), cam_label=torch.tensor([int(qc[QUERY_INDEX])], device=device), view_label=torch.tensor([0], device=device), return_feature_dict=True, semantic_masks=mask_t.unsqueeze(0).to(device) if mask_t is not None else None)
    grid = qfd.get('patch_grid', (21, 10))
    panels = [('query', denorm(img_t))]
    if mask_t is not None: panels.append(('gt_mask', color_mask(mask_t.numpy().astype(np.uint8))))
    if qfd.get('semantic_pixel_logits') is not None: panels.append(('pred_mask', color_mask(qfd['semantic_pixel_logits'][0].argmax(0).cpu().numpy().astype(np.uint8))))
    if qfd.get('patch_weights') is not None: panels.append(('patch_weights', up_heat(qfd['patch_weights'][0], grid, cfg.INPUT.SIZE_TRAIN)))
    if qfd.get('patch_reliability') is not None: panels.append(('patch_reliability', up_heat(qfd['patch_reliability'][0], grid, cfg.INPUT.SIZE_TRAIN)))
    fig, axes = plt.subplots(1, len(panels), figsize=(4*len(panels), 4))
    axes = np.atleast_1d(axes)
    for ax, (name, arr) in zip(axes, panels): ax.imshow(arr if arr.ndim == 3 else arr, cmap=None if arr.ndim == 3 else 'magma'); ax.set_title(name); ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No checkpoint available for notebook retrieval analysis.')

Recommended demo order:

1. Run the first four cells.
2. Show dataset and mask overlays.
3. Show merged config and enabled modules.
4. Run smoke forward and inspect patch/reliability/semantic outputs.
5. Run train or summarize existing logs.
6. Run test and show retrieval top-k plus query-level internals.